In [ ]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

# Suppress warnings and configure visualization style
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

def fetch_macro_data(tickers_dict, start_date="2000-01-01", end_date="2026-01-01"):
    tickers_list = list(tickers_dict.keys())
    print(f"Fetching macro data for {tickers_list}...")
    raw_df = yf.download(tickers_list, start=start_date, end=end_date, auto_adjust=True, progress=False)

    if isinstance(raw_df.columns, pd.MultiIndex):
        close_df = raw_df['Close'].copy()
    else:
        close_df = raw_df[['Close']].copy()

    close_df = close_df.rename(columns=tickers_dict)
    # Forward-fill only: back-filling leading NaNs would leak future values
    # into the warm-up period. The warm-up rows are dropped later via dropna().
    return close_df.ffill()

# Configuration
CONFIG = {
    "horizon_days": 5,
    "rsi_window": 14,
    "macd_fast": 12,
    "macd_slow": 26,
    "macd_signal": 9,
    "sma_long": 200,
    "vol_window": 21,
    "ratio_z_window": 252
}

tickers_map = {
    "GC=F": "Gold_Close",
    "DX-Y.NYB": "DXY_Close",
    "^VIX": "VIX_Close",
    "^TNX": "TNX_Close",
    "SI=F": "Silver_Close"
}

raw_data = fetch_macro_data(tickers_map)
print("Data fetched successfully.")

In [ ]:
def engineer_features(df, config=CONFIG):
    print("Engineering normalized features...")
    processed_df = df.copy()
    processed_df.sort_index(inplace=True)

    # 1. CORE GOLD INDICATORS (Normalized for XGBoost)
    delta = processed_df["Gold_Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # RSI
    com = config["rsi_window"] - 1
    ema_gain = gain.ewm(com=com, adjust=False).mean()
    ema_loss = loss.ewm(com=com, adjust=False).mean()
    processed_df["Gold_RSI"] = 100 - (100 / (1 + (ema_gain / (ema_loss + 1e-10))))

    # Normalized MACD (Scaled by price)
    exp1 = processed_df["Gold_Close"].ewm(span=config["macd_fast"], adjust=False).mean()
    exp2 = processed_df["Gold_Close"].ewm(span=config["macd_slow"], adjust=False).mean()
    macd_line = (exp1 - exp2) / processed_df["Gold_Close"]
    signal_line = macd_line.ewm(span=config["macd_signal"], adjust=False).mean()
    processed_df["Gold_MACD_Hist"] = macd_line - signal_line

    # Trend & Volatility
    processed_df["Gold_SMA_200"] = processed_df["Gold_Close"].rolling(window=config["sma_long"]).mean()
    processed_df["Dist_SMA_200"] = (processed_df["Gold_Close"] - processed_df["Gold_SMA_200"]) / processed_df["Gold_SMA_200"]
    processed_df["Gold_Vol"] = np.log(processed_df["Gold_Close"] / processed_df["Gold_Close"].shift(1)).rolling(window=config["vol_window"]).std()

    # 2. MACRO INDICATORS
    if "DXY_Close" in processed_df.columns:
        processed_df["DXY_Pct_Change"] = processed_df["DXY_Close"].pct_change()
    if "TNX_Close" in processed_df.columns:
        # ^TNX is the 10Y yield in percent (x10). diff() gives the day-over-day
        # change in that quoted value; /100 is a fixed monotonic rescale (units
        # do not matter to a tree, but keeps the feature on a small scale).
        processed_df["TNX_Diff"] = processed_df["TNX_Close"].diff() / 100.0
    if "VIX_Close" in processed_df.columns:
        processed_df["VIX_3d_Diff"] = processed_df["VIX_Close"].diff(periods=3)
    if "Silver_Close" in processed_df.columns:
        # Use a rolling z-score of the gold/silver ratio instead of the raw level.
        # The raw ratio is non-stationary and trends out of the training range,
        # which a tree model cannot extrapolate. The z-score is stationary and
        # captures how rich/cheap gold is vs silver relative to recent history.
        ratio = processed_df["Gold_Close"] / processed_df["Silver_Close"]
        roll = ratio.rolling(window=config["ratio_z_window"])
        processed_df["Gold_Silver_Ratio_Z"] = (ratio - roll.mean()) / (roll.std() + 1e-10)

    return processed_df

def build_target(df, horizon_days):
    # Future Close Price
    df["Future_Gold_Close"] = df["Gold_Close"].shift(-horizon_days)

    # Calculate exact return for Profitability tracking
    df["Forward_Return"] = (df["Future_Gold_Close"] - df["Gold_Close"]) / df["Gold_Close"]

    # Target Label: 1 if positive return, else 0
    df["Target"] = (df["Forward_Return"] > 0).astype(int)
    return df

featured_data = engineer_features(raw_data)
targeted_data = build_target(featured_data, CONFIG["horizon_days"])
cleaned_data = targeted_data.dropna().copy()
print("Features and Targets built successfully.")

In [ ]:
# Select Features
features = [
    "Gold_RSI", "Gold_MACD_Hist", "Dist_SMA_200", "Gold_Vol",
    "DXY_Pct_Change", "TNX_Diff", "VIX_3d_Diff", "Gold_Silver_Ratio_Z"
]

X = cleaned_data[features]
y = cleaned_data["Target"]
returns = cleaned_data["Forward_Return"] # Keep track of returns for the backtest

# Leakage-Free Time-Series Split
train_ratio = 0.80
split_idx = int(len(cleaned_data) * train_ratio)
purge_gap = CONFIG["horizon_days"]

# Split X and y
X_train_full = X.iloc[:split_idx]
y_train_full = y.iloc[:split_idx]

X_test = X.iloc[split_idx + purge_gap:]
y_test = y.iloc[split_idx + purge_gap:]
returns_test = returns.iloc[split_idx + purge_gap:] # Test set financial returns

# Validation split for early stopping
val_split_idx = int(len(X_train_full) * 0.85)
X_train = X_train_full.iloc[:val_split_idx]
y_train = y_train_full.iloc[:val_split_idx]

X_val = X_train_full.iloc[val_split_idx + purge_gap:]
y_val = y_train_full.iloc[val_split_idx + purge_gap:]

print(f"Training Shape: {X_train.shape}")
print(f"Validation Shape: {X_val.shape}")
print(f"Out-of-Sample Test Shape: {X_test.shape}")

In [ ]:
import xgboost as xgb

# Highly regularized model to prevent overfitting noisy financial data.
# early_stopping_rounds must be set on the constructor for the eval_set to
# actually halt training and populate best_iteration (otherwise all 1000
# trees are used and the validation set does nothing).
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.70,
    colsample_bytree=0.70,
    reg_alpha=0.5,
    reg_lambda=1.5,
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42
)

print("Training XGBoost Model...")

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# With early stopping enabled, best_iteration reflects the optimal tree count
# and predict()/predict_proba() automatically use it.
optimal_trees = xgb_model.best_iteration if getattr(xgb_model, "best_iteration", None) is not None else xgb_model.get_params()["n_estimators"]
print(f"Training complete. Optimal Trees (best_iteration): {optimal_trees}")

# Generate Predictions
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# 1. Core Metrics
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Naive baseline: always predict the majority class. Gold drifts up over the
# long run, so the "up" base rate is typically > 50%. Accuracy is only
# meaningful relative to this baseline, not relative to a coin flip.
base_rate_up = y_test.mean()
naive_accuracy = max(base_rate_up, 1 - base_rate_up)

print(f"Accuracy:            {acc:.4f}")
print(f"Naive Baseline Acc:  {naive_accuracy:.4f} (always predict majority class)")
print(f"Base Rate (Up):      {base_rate_up:.4f}")
print(f"ROC-AUC:             {roc_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# 2. Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Predicted Down/Flat", "Predicted Up"],
            yticklabels=["Actual Down/Flat", "Actual Up"])
plt.title("XGBoost Out-of-Sample Confusion Matrix", fontweight="bold")
plt.xlabel("Model Prediction")
plt.ylabel("Actual Market Movement")
plt.show()

In [ ]:
horizon = CONFIG["horizon_days"]
cost_per_trade = 0.0005  # ~5 bps charged whenever the position changes

# Assemble the full daily test frame first...
full_bt = pd.DataFrame(index=X_test.index)
full_bt["Actual_Forward_Return"] = returns_test
full_bt["Model_Prediction"] = y_pred

# ...then sample NON-OVERLAPPING periods. The 5-day forward return overlaps
# across consecutive days, so compounding daily would count each move ~5x and
# wildly overstate performance. Taking every `horizon`-th row gives disjoint
# 5-day blocks that can be compounded honestly.
backtest_df = full_bt.iloc[::horizon].copy()

# Strategy Logic:
# Prediction == 1 (Up)  -> hold gold, capture the block's Forward_Return.
# Prediction == 0 (Down/Flat) -> sit in cash (Return = 0).
backtest_df["Gross_Strategy_Return"] = backtest_df["Model_Prediction"] * backtest_df["Actual_Forward_Return"]

# Apply transaction costs whenever the position flips between blocks.
position = backtest_df["Model_Prediction"]
position_change = position.diff().abs().fillna(position.abs())
backtest_df["Strategy_Return"] = backtest_df["Gross_Strategy_Return"] - position_change * cost_per_trade

# Cumulative Returns (Growth of $1) over non-overlapping blocks
backtest_df["Cum_Benchmark"] = (1 + backtest_df["Actual_Forward_Return"]).cumprod()
backtest_df["Cum_Strategy"] = (1 + backtest_df["Strategy_Return"]).cumprod()

# Plot Profitability
plt.figure(figsize=(14, 6))
plt.plot(backtest_df.index, backtest_df["Cum_Benchmark"], label="Buy & Hold Gold (Benchmark)", color="gray", alpha=0.7)
plt.plot(backtest_df.index, backtest_df["Cum_Strategy"], label="XGBoost Strategy (Long Only, net of costs)", color="green", linewidth=2)

plt.title("Out-of-Sample Profitability: XGBoost Strategy vs Buy & Hold\n(non-overlapping 5-day periods, net of transaction costs)", fontsize=14, fontweight="bold")
plt.ylabel("Cumulative Growth (Multiplier)")
plt.xlabel("Date")
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Print Final Profitability Stats
total_bench_return = (backtest_df["Cum_Benchmark"].iloc[-1] - 1) * 100
total_strat_return = (backtest_df["Cum_Strategy"].iloc[-1] - 1) * 100

print(f"--- Profitability Results (non-overlapping {horizon}-day periods, net of {cost_per_trade*1e4:.0f} bps/trade) ---")
print(f"Number of Periods:        {len(backtest_df)}")
print(f"Buy & Hold Return:        {total_bench_return:.2f}%")
print(f"XGBoost Strategy Return:  {total_strat_return:.2f}%")

# Win Rate of trades actually taken
trades_taken = backtest_df[backtest_df["Model_Prediction"] == 1]
win_rate = (trades_taken["Actual_Forward_Return"] > 0).mean() * 100 if len(trades_taken) else float("nan")
print(f"Strategy Win Rate (When Executing): {win_rate:.2f}%")
print(f"Total Periods In-Market: {len(trades_taken)} out of {len(backtest_df)} possible periods.")